# 11 — Retrieval Benchmark (ablation study)

Notebook 10 asks *"is my system better than the obvious alternative?"* — the agent against
a plain RAG baseline. This notebook asks a different question: **"why this configuration
and not another?"**

Three design choices have been in place since the beginning without ever being measured:
`chunk_size=800`, `search_type="similarity"`, and a reranker. Each is varied here, one axis
at a time, against the same 37 scenarios.

**Scored with zero LLM calls.** Every row of `eval_scenarios.csv` carries an
`expected_evidence_keywords` phrase individually verified to exist in the target chunk's
Cause/Remedy text *and* to be absent from the question itself, so checking whether a
retrieved chunk contains it is a genuine chunk-level hit test. No generation, no judging,
no cost — and no LLM noise masking the retrieval differences being measured.

## The two metrics

| | Question it answers | Definition |
|---|---|---|
| **hit-rate@k** | *Did we find it at all?* | Does **any** of the k retrieved chunks contain the expected phrase? |
| **MRR** | *Did we find it at the top?* | Mean of 1/position of the first correct chunk. 1st → 1.00, 2nd → 0.50, 5th → 0.20, absent → 0 |

Both are needed. Two configurations can share a hit-rate while one leads with the answer
and the other buries it fifth — the model weights what comes first, so only MRR sees that.

## One deliberate exclusion: `code_aware` is OFF throughout

The `CodeAwareRetriever` (see `dificuldades_e_oportunidades.md` #1) resolves fault codes by
exact lexical lookup. With it on, the 17 `vfd_fault_code` scenarios out of 37 would score
perfectly under **every** configuration and flatten the very differences this study exists
to measure. Its own gain is already quantified separately (definition page 18% → 100%);
nothing is lost by isolating it here.

In [1]:
import sys, time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import COLLECTION_NAME, VECTOR_DIR
from factory_floor.rag import build_retriever, get_llm
from factory_floor.retrieval_benchmark import (
    build_variant_store, evaluate_retriever, format_table,
    load_eval_scenarios, variant_chunk_count, variant_dir,
)
from factory_floor.vectorstore import get_embeddings, load_vectorstore

embeddings = get_embeddings()
scenarios = load_eval_scenarios()
production_store = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)

print(f'{len(scenarios)} scenarios')
print(f'production store (chunk_size=800): {variant_chunk_count(production_store)} chunks')

BASELINE = dict(k=5, rerank=False, code_aware=False, search_type='similarity')

def factory_for(store, **overrides):
    """Each scenario carries its own equipment_type filter, exactly as
    evaluation.run_baseline() applies it -- comparing configurations without honouring
    that filter would measure something the app never runs."""
    config = {**BASELINE, **overrides}
    return lambda equipment_type: build_retriever(store, equipment_type=equipment_type, **config)

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


37 scenarios
production store (chunk_size=800): 12301 chunks


## Axis A — chunk size

Rebuilds the corpus at 400 and 1600 characters (overlap held at 15% of chunk size) into
**their own directories**. This is a safety requirement, not tidiness: `build_vectorstore`'s
`rebuild=True` calls `shutil.rmtree()` on the directory it is given, so pointing a variant at
the production `VECTOR_DIR` — even under a different collection name — would delete the real
store.

Expect a few minutes and roughly 200-300 MB per variant on first run. The helper is
idempotent, so re-running the notebook reuses what is already on disk.

The tension being measured: small chunks are precise but sever context (a fault's *Cause*
ends up in a different chunk from its *Remedy*); large chunks preserve context but dilute the
signal with unrelated neighbouring text.

In [2]:
stores = {800: production_store}
for chunk_size in (400, 1600):
    t0 = time.time()
    stores[chunk_size] = build_variant_store(chunk_size, embeddings=embeddings)
    print(f'chunk_size={chunk_size}: {variant_chunk_count(stores[chunk_size])} chunks '
          f'({time.time()-t0:.0f}s)')

axis_a = {}
rows = []
for chunk_size in sorted(stores):
    result = evaluate_retriever(factory_for(stores[chunk_size]), scenarios)
    axis_a[chunk_size] = result
    marker = '  <- current' if chunk_size == 800 else ''
    rows.append((f'chunk_size={chunk_size} ({variant_chunk_count(stores[chunk_size])} chunks)',
                 result['hit_rate'], result['mrr'], marker))

print()
print(format_table('AXIS A - chunk size (similarity, k=5, no reranker)', rows))

chunk_size=400: 23336 chunks (0s)


chunk_size=1600: 6754 chunks (0s)



AXIS A - chunk size (similarity, k=5, no reranker)
--------------------------------------------------------------------
Configuration                       hit-rate@k       MRR            
chunk_size=400 (23336 chunks)           64.9%     0.433            
chunk_size=800 (12301 chunks)           78.4%     0.596  <- current
chunk_size=1600 (6754 chunks)           81.1%     0.610            


In [3]:
best_chunk_size = max(axis_a, key=lambda cs: (axis_a[cs]['mrr'], axis_a[cs]['hit_rate']))
print(f'Best by MRR: chunk_size={best_chunk_size}\n')

print(f"{'category':<20}" + ''.join(f'{cs:>16}' for cs in sorted(axis_a)))
categories = sorted(axis_a[800]['per_category'])
for category in categories:
    line = f'{category:<20}'
    for chunk_size in sorted(axis_a):
        m = axis_a[chunk_size]['per_category'].get(category, {'hit_rate': 0, 'mrr': 0})
        line += f"{m['hit_rate']:>9.0%}/{m['mrr']:>5.2f}"
    print(line)
print('\n(hit-rate / MRR per cell)')

Best by MRR: chunk_size=1600

category                         400             800            1600
general                   67%/ 0.19      67%/ 0.40     100%/ 0.57
motor_manual              75%/ 0.55      75%/ 0.62      75%/ 0.62
motor_scenario            80%/ 0.50      80%/ 0.62      80%/ 0.68
vfd_fault_code            59%/ 0.43      82%/ 0.63      82%/ 0.62
vfd_general               33%/ 0.33      67%/ 0.50      67%/ 0.33

(hit-rate / MRR per cell)


## Axis B — search strategy

Run on the best chunking from axis A.

- **similarity** — the k nearest neighbours. What the project has always used.
- **mmr** (maximal marginal relevance) — trades some relevance for diversity, penalising
  chunks that repeat one another. Plausibly useful here, since the List Manual pages are
  highly repetitive.
- **similarity_score_threshold** — returns only chunks above a distance threshold, and
  therefore *fewer than k* (or none) when nothing is close enough. This is the only option
  that can address difficulty #2, where an off-topic question still fills the sources table.

⚠️ Threshold caveat, worth stating rather than hiding: the collection is built with
`hnsw.space='l2'`, so LangChain resolves the threshold through its Euclidean relevance
function. The number below is **not** a cosine similarity and is not comparable to
thresholds quoted for cosine-space collections.

In [4]:
best_store = stores[best_chunk_size]
axis_b = {}
rows = []

configs_b = [
    ('similarity', {}),
    ('mmr (lambda_mult=0.5)', {'search_type': 'mmr', 'search_kwargs_extra': {'lambda_mult': 0.5}}),
    ('similarity_score_threshold (0.2)',
     {'search_type': 'similarity_score_threshold', 'search_kwargs_extra': {'score_threshold': 0.2}}),
]
for label, overrides in configs_b:
    result = evaluate_retriever(factory_for(best_store, **overrides), scenarios)
    axis_b[label] = result
    mean_docs = sum(r['n_documents'] for r in result['results']) / len(result['results'])
    rows.append((label, result['hit_rate'], result['mrr'], f'{mean_docs:.1f} docs'))

print(format_table(f'AXIS B - search strategy (chunk_size={best_chunk_size}, k=5, no reranker)', rows))
print()
print('The rightmost column is the mean number of chunks actually returned -- only the')
print('threshold strategy can return fewer than k.')
print()
print('!! The threshold row is NOT a valid comparison. LangChain emitted')
print('   "Relevance scores must be between 0 and 1" with NEGATIVE values, because this')
print('   collection is built with hnsw.space=l2 and the Euclidean relevance function')
print('   does not produce a 0-1 score here. The threshold therefore discards almost')
print('   everything regardless of relevance. Read that row as "this metric is broken for')
print('   this collection", not as "thresholding retrieves worse".')

/opt/miniconda3/lib/python3.13/site-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='37ffe2bd-b410-46a0-a529-cdf040a8f676', metadata={'documentid': 'A5E38483075A', 'language': 'en', 'subject': 'Low-Voltage Motors', 'chunk_id': 2457, 'moddate': '2023-12-05T09:14:55+01:00', 'swversion': '', 'equipment_type': 'electric_motor', 'producer': 'Antenna House PDF Output Library 7.2.1790; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA', 'setdate': '2023-12-05', 'articlenumber': '1LE1;1FP1;1FP3;1PC1;1PC3', 'page': 94, 'trapped': 'False', 'manufacturer': 'Siemens', 'source_file': 'Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf', 'author': 'Innomotics\xa0GmbH', 'documentversion': '13.00', 'summary': '', 'hwversion': '', 'creationdate': '2023-12-05T09:13:48+01:00', 'source': '/Users/marcelocorreia/Desktop/Factory_Floor_Chatbot/data/manuals/Siemens_SIMOTICS_GP_1LE1_Operating_Instructions.pdf', 'total_pages': 140, 'validi

No relevant docs were retrieved using the relevance score threshold 0.2


No relevant docs were retrieved using the relevance score threshold 0.2


No relevant docs were retrieved using the relevance score threshold 0.2


No relevant docs were retrieved using the relevance score threshold 0.2


No relevant docs were retrieved using the relevance score threshold 0.2


No relevant docs were retrieved using the relevance score threshold 0.2


AXIS B - search strategy (chunk_size=1600, k=5, no reranker)
--------------------------------------------------------------------
Configuration                       hit-rate@k       MRR            
similarity                              81.1%     0.610    5.0 docs
mmr (lambda_mult=0.5)                   73.0%     0.591    5.0 docs
similarity_score_threshold (0.2)        62.2%     0.452    3.8 docs

The rightmost column is the mean number of chunks actually returned -- only the
threshold strategy can return fewer than k.

!! The threshold row is NOT a valid comparison. LangChain emitted
   "Relevance scores must be between 0 and 1" with NEGATIVE values, because this
   collection is built with hnsw.space=l2 and the Euclidean relevance function
   does not produce a 0-1 score here. The threshold therefore discards almost
   everything regardless of relevance. Read that row as "this metric is broken for
   this collection", not as "thresholding retrieves worse".


## Axis C — the reranker

The reranker fetches a wider candidate pool (`fetch_k = max(k*3, 12)`) and asks one LLM call
to reorder it, keeping the top k. It has been enabled in `app.py` since it was written, and
never measured until now.

This is the one axis that costs API calls: one per scenario, 37 in total.

Note it does two things at once, which matters when reading the result: it **widens** the
candidate pool *and* **reorders** it. So an improvement in hit-rate comes from the wider
pool, while an improvement in MRR is the reordering doing real work. Axis D below provides
the control that separates the two.

In [5]:
llm = get_llm()
axis_c = {}
rows = []
for label, overrides in [('reranker OFF', {}), ('reranker ON', {'rerank': True, 'rerank_llm': llm})]:
    result = evaluate_retriever(factory_for(best_store, **overrides), scenarios)
    axis_c[label] = result
    rows.append((label, result['hit_rate'], result['mrr'], f"{result['elapsed_s']:.0f}s"))

print(format_table(f'AXIS C - reranker (chunk_size={best_chunk_size}, similarity, k=5)', rows))

delta_hit = axis_c['reranker ON']['hit_rate'] - axis_c['reranker OFF']['hit_rate']
delta_mrr = axis_c['reranker ON']['mrr'] - axis_c['reranker OFF']['mrr']
print(f'\nDelta: hit-rate {delta_hit:+.1%}, MRR {delta_mrr:+.3f}')

AXIS C - reranker (chunk_size=1600, similarity, k=5)
--------------------------------------------------------------------
Configuration                       hit-rate@k       MRR            
reranker OFF                            81.1%     0.610          7s
reranker ON                             86.5%     0.695         43s

Delta: hit-rate +5.4%, MRR +0.085


## Axis D — k

How many chunks to retrieve. Larger k almost mechanically raises hit-rate (more chances to
contain the phrase) while diluting the context the model has to read, so hit-rate alone is
misleading here — MRR and the hit-rate-per-chunk-read trade-off are what matter.

In [6]:
axis_d = {}
rows = []
for k in (3, 5, 10):
    result = evaluate_retriever(factory_for(best_store, k=k), scenarios)
    axis_d[k] = result
    marker = '  <- current' if k == 5 else ''
    rows.append((f'k={k}', result['hit_rate'], result['mrr'], marker))

print(format_table(f'AXIS D - k (chunk_size={best_chunk_size}, similarity, no reranker)', rows))

AXIS D - k (chunk_size=1600, similarity, no reranker)
--------------------------------------------------------------------
Configuration                       hit-rate@k       MRR            
k=3                                     75.7%     0.599            
k=5                                     81.1%     0.610  <- current
k=10                                    86.5%     0.619            


### Is the reranker just "fetch more"?

The reranker pulls 15 candidates and returns the best 5. Simply setting `k=10` also pulls
more. If both reach the same hit-rate, the honest question is whether the LLM call is buying
anything beyond the wider pool — and the answer is in the MRR and in how much context the
model then has to read.

In [7]:
rerank_on = axis_c['reranker ON']
k10 = axis_d[10]
print(f"{'':<34}{'hit-rate':>11}{'MRR':>10}{'chunks read':>14}")
print(f"{'reranker ON (k=5, fetches 15)':<34}{rerank_on['hit_rate']:>10.1%}{rerank_on['mrr']:>10.3f}{5:>14}")
print(f"{'plain k=10 (no reranker)':<34}{k10['hit_rate']:>10.1%}{k10['mrr']:>10.3f}{10:>14}")
print()
print(f'Same coverage' if abs(rerank_on['hit_rate'] - k10['hit_rate']) < 0.001
      else f"Coverage differs by {rerank_on['hit_rate'] - k10['hit_rate']:+.1%}", end='')
print(f", but MRR {rerank_on['mrr'] - k10['mrr']:+.3f} for the reranker, on half the context.")

                                     hit-rate       MRR   chunks read
reranker ON (k=5, fetches 15)          86.5%     0.695             5
plain k=10 (no reranker)               86.5%     0.619            10

Same coverage, but MRR +0.076 for the reranker, on half the context.


## Summary and honest limitations

## Does combining the per-axis winners actually win?

An ablation measures each axis **in isolation**. Stacking the winners assumes the effects
add up, which is not guaranteed — so the combination is measured directly rather than
inferred, against the configuration actually shipping today.

In [8]:
combined = evaluate_retriever(
    factory_for(stores[best_chunk_size], rerank=True, rerank_llm=llm), scenarios)
current = axis_a[800]

rows = [
    ('CURRENT production (800/sim/k=5)', current['hit_rate'], current['mrr'], ''),
    (f'COMBINED ({best_chunk_size}/sim/k=5/rerank)', combined['hit_rate'], combined['mrr'], ''),
]
print(format_table('COMBINED vs CURRENT', rows))
print(f"\nDelta: hit-rate {combined['hit_rate'] - current['hit_rate']:+.1%}, "
      f"MRR {combined['mrr'] - current['mrr']:+.3f}")
print()
print('Note: production also runs code_aware=True, which is off throughout this notebook')
print('and already lifts fault-code retrieval to 100% on its own. These numbers describe')
print('the semantic path only.')

COMBINED vs CURRENT
--------------------------------------------------------------------
Configuration                       hit-rate@k       MRR            
CURRENT production (800/sim/k=5)        78.4%     0.596            
COMBINED (1600/sim/k=5/rerank)          86.5%     0.695            

Delta: hit-rate +8.1%, MRR +0.098

Note: production also runs code_aware=True, which is off throughout this notebook
and already lifts fault-code retrieval to 100% on its own. These numbers describe
the semantic path only.


In [9]:
print('WINNER PER AXIS (by MRR, tie-broken on hit-rate)\n')
print(f"  A chunk size      : {best_chunk_size}")
best_b = max(axis_b, key=lambda x: (axis_b[x]['mrr'], axis_b[x]['hit_rate']))
best_c = max(axis_c, key=lambda x: (axis_c[x]['mrr'], axis_c[x]['hit_rate']))
best_d = max(axis_d, key=lambda x: (axis_d[x]['mrr'], axis_d[x]['hit_rate']))
print(f'  B search strategy : {best_b}')
print(f'  C reranker        : {best_c}')
print(f'  D k               : {best_d}')

current = axis_a[800]
print(f"\nCurrent production config (800 / similarity / k=5): "
      f"hit-rate {current['hit_rate']:.1%}, MRR {current['mrr']:.3f}")

WINNER PER AXIS (by MRR, tie-broken on hit-rate)

  A chunk size      : 1600
  B search strategy : similarity
  C reranker        : reranker ON
  D k               : 10

Current production config (800 / similarity / k=5): hit-rate 78.4%, MRR 0.596


### Limitations — read these before quoting any number above

1. **A phrase match is a proxy for relevance, not relevance itself.** A chunk can contain the
   expected phrase and still be the wrong page, and a genuinely useful chunk phrased
   differently counts as a miss. Same proxy limitation already disclosed for notebook 10's
   `keyword_score`, applied one layer earlier in the pipeline.
2. **37 scenarios is a small sample.** Differences of a few percentage points between
   configurations are within noise; only large gaps should drive a decision.
3. **The category breakdown is smaller still** — `general` and `vfd_general` have 3 scenarios
   each, so their per-category figures move in 33% steps and should not be read as trends.
4. **The `similarity_score_threshold` result is invalid, not merely caveated.** This
   collection uses `hnsw.space='l2'`, and LangChain's Euclidean relevance function
   returned scores outside 0-1 (several negative), so the threshold discarded chunks
   irrespective of relevance. That row measures a broken metric, not a worse strategy.
   Testing thresholding properly would require rebuilding the collection in cosine space.
5. **This measures retrieval, not answers.** A better-retrieving configuration should produce
   better answers, but that link is assumed here, not demonstrated — confirming it end to end
   would mean re-running notebook 10 per configuration.
6. **`code_aware` is off throughout**, so the `vfd_fault_code` figures describe the *semantic*
   path only. The shipped app resolves those codes by exact lookup instead.

### Cleaning up

The variant stores are only needed to re-run this notebook. To reclaim the disk:

```bash
rm -rf data/vectorstore_cs400 data/vectorstore_cs1600
```

`data/vectorstore/` is the production store and is never written to by this notebook.